<a href="https://colab.research.google.com/github/JonTheBruce/infsys/blob/main/rag_chat_colab_Jon_Bruce_week9.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# RAG Chatbot with PDFs (Colab-ready)

This notebook builds a simple RAG chatbot over PDF files:

1. Install dependencies  
2. Upload PDFs into a local `pdfs/` folder  
3. Download a small local Llama model  
4. Extract PDF text  
5. Chunk and embed the text  
6. Build a FAISS vector index  
7. Ask questions using retrieved context

Run the cells from top to bottom in Google Colab.


In [1]:
# Install dependencies
# This may take several minutes, especially llama-cpp-python.
!pip -q install -U langchain-text-splitters langchain langchain-community sentence-transformers faiss-cpu pypdf tiktoken llama-cpp-python==0.2.90


     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 63.8/63.8 MB 11.3 MB/s eta 0:00:00
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Installing backend dependencies ... done
  Preparing metadata (pyproject.toml) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.4/2.4 MB 61.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 740.6/740.6 kB 35.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.8/18.8 MB 69.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 405.1/405.1 kB 22.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 45.5/45.5 kB 3.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 44.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 80.4/80.4 kB 5.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 73.1/73.1 kB 4.4 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This 

In [2]:
# Imports and folders
import glob
import os
import urllib.request

MODEL_DIR = "./models"
PDF_DIR = "./pdfs"

os.makedirs(MODEL_DIR, exist_ok=True)
os.makedirs(PDF_DIR, exist_ok=True)

print("Folders ready:")
print("MODEL_DIR:", MODEL_DIR)
print("PDF_DIR:", PDF_DIR)


Folders ready:
MODEL_DIR: ./models
PDF_DIR: ./pdfs


## Upload PDFs

Use the next cell to upload one or more PDF files from your computer.  
The uploaded files will be saved into the `./pdfs` folder.


In [4]:
# Upload PDFs in Colab
from google.colab import files

uploaded = files.upload()

for filename, filedata in uploaded.items():
    if filename.lower().endswith(".pdf"):
        save_path = os.path.join(PDF_DIR, filename)
        with open(save_path, "wb") as f:
            f.write(filedata)
        print("Saved:", save_path)
    else:
        print("Skipped non-PDF file:", filename)


Saving Shipping_Guide.pdf to Shipping_Guide.pdf
Saving Privacy_Notice.pdf to Privacy_Notice.pdf
Saving Warranty_Terms.pdf to Warranty_Terms.pdf
Saving Return_Policy.pdf to Return_Policy.pdf
Saving Service_Agreement.pdf to Service_Agreement.pdf
Saved: ./pdfs/Shipping_Guide.pdf
Saved: ./pdfs/Privacy_Notice.pdf
Saved: ./pdfs/Warranty_Terms.pdf
Saved: ./pdfs/Return_Policy.pdf
Saved: ./pdfs/Service_Agreement.pdf


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [5]:
# Download the local LLM model
# TinyLlama 1.1B Chat Q4_K_M is about 0.6-0.8 GB.
MODEL_URL = "https://huggingface.co/TheBloke/TinyLlama-1.1B-Chat-v1.0-GGUF/resolve/main/tinyllama-1.1b-chat-v1.0.Q4_K_M.gguf?download=true"
MODEL_PATH = os.path.join(MODEL_DIR, "tinyllama-1.1b-chat-v1.0.Q4_K_M.gguf")

if not os.path.exists(MODEL_PATH):
    print("Downloading model... this may take a few minutes.")
    urllib.request.urlretrieve(MODEL_URL, MODEL_PATH)
else:
    print("Model already present:", MODEL_PATH)

print("MODEL_PATH:", MODEL_PATH)


MODEL_PATH: ./models/tinyllama-1.1b-chat-v1.0.Q4_K_M.gguf


In [6]:
# Find PDFs
pdf_files = sorted(glob.glob(os.path.join(PDF_DIR, "*.pdf")))
print("Found PDFs:", [os.path.basename(p) for p in pdf_files] if pdf_files else "None yet")


Found PDFs: ['Privacy_Notice.pdf', 'Return_Policy.pdf', 'Service_Agreement.pdf', 'Shipping_Guide.pdf', 'Warranty_Terms.pdf']


In [7]:
# Load PDF text
from pypdf import PdfReader

def load_pdfs(pdf_paths):
    texts = []

    for path in pdf_paths:
        try:
            reader = PdfReader(path)
            pages = []

            for page in reader.pages:
                txt = page.extract_text() or ""
                if txt.strip():
                    pages.append(txt)

            joined = "\n".join(pages).strip()

            if joined:
                texts.append((os.path.basename(path), joined))
                print(f"Loaded: {os.path.basename(path)} ({len(joined)} chars)")
            else:
                print(f"Warning: {os.path.basename(path)} has no extractable text.")

        except Exception as e:
            print(f"Failed to read {path}: {e}")

    return texts

raw_texts = load_pdfs(pdf_files)

if not raw_texts:
    print("No PDF text loaded. Upload PDFs and re-run from the PDF search/loading cells.")


Loaded: Privacy_Notice.pdf (459 chars)
Loaded: Return_Policy.pdf (462 chars)
Loaded: Service_Agreement.pdf (708 chars)
Loaded: Shipping_Guide.pdf (479 chars)
Loaded: Warranty_Terms.pdf (450 chars)


In [8]:
# Chunk the PDF text
from langchain_text_splitters import RecursiveCharacterTextSplitter

def chunk_texts(name_and_text, chunk_size=800, chunk_overlap=150):
    splitter = RecursiveCharacterTextSplitter(
        chunk_size=chunk_size,
        chunk_overlap=chunk_overlap,
        separators=["\n\n", "\n", ". ", " ", ""],
    )

    docs = []

    for fname, text in name_and_text:
        for chunk in splitter.split_text(text):
            docs.append({"source": fname, "text": chunk})

    print(f"Created {len(docs)} chunks.")
    return docs

docs = chunk_texts(raw_texts) if raw_texts else []


Created 5 chunks.


In [9]:
# Build embeddings and FAISS vector index
from langchain_community.embeddings import HuggingFaceEmbeddings
from langchain_community.vectorstores import FAISS

EMBED_MODEL = "sentence-transformers/all-MiniLM-L6-v2"
embedder = HuggingFaceEmbeddings(model_name=EMBED_MODEL)

if docs:
    texts = [d["text"] for d in docs]
    metadatas = [{"source": d["source"]} for d in docs]

    vectordb = FAISS.from_texts(
        texts=texts,
        embedding=embedder,
        metadatas=metadatas
    )

    retriever = vectordb.as_retriever(search_kwargs={"k": 4})
    print("FAISS index built.")
else:
    vectordb = None
    retriever = None
    print("No chunks available, so no index was built.")


/tmp/ipykernel_1366/993205401.py:2: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.embeddings import HuggingFaceEmbeddings
/tmp/ipykernel_1366/993205401.py:6: LangChainDeprecationWarning: The class `HuggingFaceEmbeddings` was deprecated in LangChain 0.2.2 and will be removed in 1.0. An updated version of the class exists in the `langchain-huggingface package and should be used instead. To use it run `pip install -U `langchain-huggingface` and import as `from `langchain_huggingface import HuggingFaceEmbeddings``.
  embedder = HuggingFaceEmbeddings(model_name=EMBED_MODEL)


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/10.5k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

...

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

FAISS index built.


In [10]:
# Load the local LLM
from langchain_community.chat_models import ChatLlamaCpp
from langchain_core.messages import HumanMessage, SystemMessage

LLM = ChatLlamaCpp(
    model_path=MODEL_PATH,
    temperature=0,
    max_tokens=512,
    n_ctx=4096,
    verbose=False,
)

print("LLM loaded.")


LLM loaded.


In [11]:
# RAG question-answering function
def _format_context(docs):
    return "\n\n".join(
        [f"[From: {d.metadata.get('source', '?')}]\n{d.page_content}" for d in docs]
    )

def answer_question(question: str, k: int = 4) -> str:
    if retriever is None:
        return "No documents available. Please upload PDFs and rebuild the index."

    # Retrieve relevant chunks
    top_docs = retriever.invoke(question)

    system_prompt = (
        "Answer ONLY using the provided context. "
        "If the answer is not in the context, say you don't know."
    )

    user_prompt = f"""Context:
{_format_context(top_docs)}

Question: {question}

Answer in 3-6 sentences and cite file names when helpful."""

    messages = [
        SystemMessage(content=system_prompt),
        HumanMessage(content=user_prompt),
    ]

    resp = LLM.invoke(messages)
    return resp.content if hasattr(resp, "content") else str(resp)


In [12]:
# Ask a question
answer_question("What is the warranty period?")


'The warranty period for SmartHome Co.\'s products includes a one-year limited warranty covering manufacturing defects. The warranty does not cover water damage, unauthorized repairs, or modifications. Customers can file claims through support@smarthomeco.com with proof of purchase and a photo of the issue. SmartHome Co. Reserves the right to refuse service, terminate accounts, or seek legal remedies for violations of the warranty terms. The return policy allows customers to return items within 30 days of purchase with proof of receipt, and refunds are processed within 10 business days to the original payment method. SmartHome Co.\'s shipping guide provides information on domestic and international shipping, including estimated delivery times and eligibility for replacement shipments. The warranty period is covered by the file names "SmartHome Co. Warranty Terms" (PDF), "SmartHome Co. Service Agréement" (PDF), "SmartHome Co. Return Policy" (PDF), and "SmartHome Co. Shipping Guide" (PDF

In [13]:
# Ask a question
answer_question("How do I reset my device?")


'To reset your device, follow these steps:\n\n1. Turn off your device.\n2. Locate the reset button or code on your device. This may be located on the back panel or in a dedicated reset location.\n3. Press and hold the reset button or code for 10-15 seconds until the device beeps three times.\n4. Release the reset button or code.\n5. Wait for the device to restart.\n6. If your device does not restart, try resetting it again using the same method.\n7. Once your device has been reset, follow the instructions provided by SmartHome Co. For any further assistance.'